# Phase C - Harm Classification

This notebook trains and evaluates the multi-label harm classifier on the annotated posts. The three-tier harm labels are collapsed to binary (baseline vs elevated) because the severe tiers are too sparse to model. Three variants are compared - a lexicon-only baseline, the domain-adapted encoder, and a combined model - a hyperparameter sweep selects the configuration, and the best model is retrained on all data for the linkage analysis in Notebook 09.

**Input:** the annotated set with binary targets. **Output:** the final classifier and its test-set predictions.

# 1. Setup and Cleaning Labels from annotation

In [ ]:
import pandas as pd
d = pd.read_csv("data/processed/to_annotate.csv")

# fix invalid cells
d.loc[d.text=="must protect at all costs", "parasocial_risk"] = "moderate"
d.loc[d.text=="atinys are in some weird shit", "bullying"] = "involved"
d = d[d.type != "m"].copy()   # drop parsing artifact

# add binary targets for modelling (keep fine labels for descriptive reporting)
d["y_para"] = (d.parasocial_risk.str.strip().str.lower()!="none").astype(int)
d["y_bully"] = (d.bullying.str.strip().str.lower()!="none").astype(int)
d["y_fin"] = (d.financial_harm.str.strip().str.lower()!="none").astype(int)

d.to_parquet("data/processed/labelled.parquet")
print("labelled:", len(d))
print("positives — para:", d.y_para.sum(), "bully:", d.y_bully.sum(), "fin:", d.y_fin.sum())

In [2]:
import os, sys
import numpy as np, pandas as pd
import torch
os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
sys.path.append("src")
from clf import FocalLoss, Posts, HarmClassifier, score

PROC = "data/processed"
d = pd.read_csv(f"{PROC}/to_annotate.csv")

# fix the 2 invalid cells + drop the parsing artifact
d.loc[d.text=="must protect at all costs", "parasocial_risk"] = "moderate"
d.loc[d.text=="atinys are in some weird shit", "bullying"] = "involved"
d = d[d.type != "m"].copy()

for c in ["parasocial_risk","bullying","financial_harm"]:
    d[c] = d[c].fillna("none").astype(str).str.strip().str.lower()

# binary targets: none vs elevated
d["y_para"]  = (d.parasocial_risk != "none").astype(int)
d["y_bully"] = (d.bullying != "none").astype(int)
d["y_fin"]   = (d.financial_harm != "none").astype(int)
Y = d[["y_para","y_bully","y_fin"]].values.astype(float)

d.to_parquet(f"{PROC}/labelled.parquet")
print("posts:", len(d), "| positives:", d[["y_para","y_bully","y_fin"]].sum().to_dict())

posts: 2495 | positives: {'y_para': 249, 'y_bully': 244, 'y_fin': 98}


# Train/Test Split

In [3]:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(np.arange(len(d)), test_size=0.2,
                          random_state=0, stratify=d.y_bully)
print("train:", len(tr), "test:", len(te), "| test positives:", Y[te].sum(0))

train: 1996 test: 499 | test positives: [52. 49. 23.]


# VARIANT A: lexicon-only baseline (CPU)

In [10]:
scored = pd.read_parquet(f"{PROC}/posts_scored.parquet")
lexcols = [c for c in scored.columns if c.startswith(("parasocial_","financial_","victim_"))] \
          + ["caps","excl","first_person"]
lex = d.merge(scored[["text"]+lexcols].drop_duplicates("text"), on="text", how="left").fillna(0)
XL = lex[lexcols].values

from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
base = MultiOutputClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
base.fit(XL[tr], Y[tr])
probA = np.column_stack([e.predict_proba(XL[te])[:,1] for e in base.estimators_])
mA = score(Y[te], probA)
print("=== A: lexicon-only ==="); [print(" ", k, v) for k,v in mA.items()]

=== A: lexicon-only ===
  parasocial {'precision': 0.168, 'recall': 0.635, 'f1': 0.265, 'auc': 0.661, 'support': 52}
  bullying {'precision': 0.151, 'recall': 0.878, 'f1': 0.257, 'auc': 0.769, 'support': 49}
  financial {'precision': 0.103, 'recall': 0.478, 'f1': 0.169, 'auc': 0.754, 'support': 23}


[None, None, None]

# VARIANT B: DAPT fine-tune (GPU)

In [11]:
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader

tok = AutoTokenizer.from_pretrained("models/dapt_final")
texts = d.text.tolist()

def loader(idx, bs=16, shuffle=False):
    return DataLoader(Posts([texts[i] for i in idx], Y[idx], tok),
                      batch_size=bs, shuffle=shuffle)

model = HarmClassifier(AutoModel.from_pretrained("models/dapt_final")).cuda()

pos = Y[tr].sum(0); pw = torch.tensor((len(tr)-pos)/np.maximum(pos,1),
                                       dtype=torch.float).cuda()
loss_fn = FocalLoss(gamma=2.0, pos_weight=pw)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(4):
    model.train(); tot=0
    for ids, mask, y in loader(tr, shuffle=True):
        opt.zero_grad()
        loss = loss_fn(model(ids.cuda(), mask.cuda()), y.cuda())
        loss.backward(); opt.step(); tot += loss.item()
    print(f"epoch {epoch+1}: loss {tot/len(loader(tr)):.4f}")

model.eval(); probs=[]
with torch.no_grad():
    for ids, mask, y in loader(te):
        probs.append(torch.sigmoid(model(ids.cuda(), mask.cuda())).cpu().numpy())
probB = np.vstack(probs)
mB = score(Y[te], probB)
print("=== B: DAPT ==="); [print(" ", k, v) for k,v in mB.items()]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: loss 0.2742
epoch 2: loss 0.2100
epoch 3: loss 0.1759
epoch 4: loss 0.1297
=== B: DAPT ===
  parasocial {'precision': 0.206, 'recall': 0.635, 'f1': 0.311, 'auc': 0.729, 'support': 52}
  bullying {'precision': 0.469, 'recall': 0.612, 'f1': 0.531, 'auc': 0.884, 'support': 49}
  financial {'precision': 0.247, 'recall': 0.826, 'f1': 0.38, 'auc': 0.867, 'support': 23}


[None, None, None]

In [12]:
import numpy as np
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    m = score(Y[te], probB, thresh=t)
    macro_f1 = np.mean([m[k]["f1"] for k in m])
    print(f"thresh {t}: macro-F1 {macro_f1:.3f} | "
          f"para P/R {m['parasocial']['precision']:.2f}/{m['parasocial']['recall']:.2f} | "
          f"bully P/R {m['bullying']['precision']:.2f}/{m['bullying']['recall']:.2f} | "
          f"fin P/R {m['financial']['precision']:.2f}/{m['financial']['recall']:.2f}")

thresh 0.3: macro-F1 0.331 | para P/R 0.12/0.90 | bully P/R 0.35/0.80 | fin P/R 0.18/0.83
thresh 0.4: macro-F1 0.367 | para P/R 0.16/0.83 | bully P/R 0.39/0.69 | fin P/R 0.21/0.83
thresh 0.5: macro-F1 0.407 | para P/R 0.21/0.64 | bully P/R 0.47/0.61 | fin P/R 0.25/0.83
thresh 0.6: macro-F1 0.379 | para P/R 0.28/0.48 | bully P/R 0.43/0.37 | fin P/R 0.27/0.70
thresh 0.7: macro-F1 0.283 | para P/R 0.23/0.14 | bully P/R 0.44/0.16 | fin P/R 0.33/0.65
thresh 0.8: macro-F1 0.146 | para P/R 0.00/0.00 | bully P/R 0.50/0.02 | fin P/R 0.41/0.39


# Cell 5 — VARIANT C: combined (DAPT + lexicon)

In [13]:
from sklearn.preprocessing import StandardScaler
XLs = StandardScaler().fit(XL[tr]).transform(XL)

model2 = HarmClassifier(AutoModel.from_pretrained("models/dapt_final"),
                        extra_dim=XLs.shape[1]).cuda()
opt2 = torch.optim.AdamW(model2.parameters(), lr=2e-5)
loss_fn2 = FocalLoss(gamma=2.0, pos_weight=pw)

def batches(idx, bs=16, shuffle=True):
    idx = np.array(idx)
    if shuffle: np.random.shuffle(idx)
    for i in range(0, len(idx), bs): yield idx[i:i+bs]

for epoch in range(4):
    model2.train(); tot=0; nb=0
    for bi in batches(tr):
        e = tok([texts[i] for i in bi], truncation=True, max_length=192,
                padding=True, return_tensors="pt")
        ex = torch.tensor(XLs[bi], dtype=torch.float).cuda()
        y = torch.tensor(Y[bi], dtype=torch.float).cuda()
        opt2.zero_grad()
        loss = loss_fn2(model2(e.input_ids.cuda(), e.attention_mask.cuda(), extra=ex), y)
        loss.backward(); opt2.step(); tot+=loss.item(); nb+=1
    print(f"epoch {epoch+1}: loss {tot/nb:.4f}")

model2.eval(); probs=[]
with torch.no_grad():
    for bi in batches(te, shuffle=False):
        e = tok([texts[i] for i in bi], truncation=True, max_length=192,
                padding=True, return_tensors="pt")
        ex = torch.tensor(XLs[bi], dtype=torch.float).cuda()
        probs.append(torch.sigmoid(
            model2(e.input_ids.cuda(), e.attention_mask.cuda(), extra=ex)).cpu().numpy())
probC = np.vstack(probs)
mC = score(Y[te], probC)
print("=== C: combined ==="); [print(" ", k, v) for k,v in mC.items()]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: loss 0.2920
epoch 2: loss 0.2139
epoch 3: loss 0.1694
epoch 4: loss 0.1121
=== C: combined ===
  parasocial {'precision': 0.185, 'recall': 0.096, 'f1': 0.127, 'auc': 0.634, 'support': 52}
  bullying {'precision': 0.33, 'recall': 0.714, 'f1': 0.452, 'auc': 0.886, 'support': 49}
  financial {'precision': 0.308, 'recall': 0.696, 'f1': 0.427, 'auc': 0.85, 'support': 23}


[None, None, None]

from sklearn.preprocessing import StandardScaler
XLs = StandardScaler().fit(XL[tr]).transform(XL)

model2 = HarmClassifier(AutoModel.from_pretrained("models/dapt_final"),
                        extra_dim=XLs.shape[1]).cuda()
opt2 = torch.optim.AdamW(model2.parameters(), lr=2e-5)
loss_fn2 = FocalLoss(gamma=2.0, pos_weight=pw)

def batches(idx, bs=16, shuffle=True):
    idx = np.array(idx)
    if shuffle: np.random.shuffle(idx)
    for i in range(0, len(idx), bs): yield idx[i:i+bs]

for epoch in range(4):
    model2.train(); tot=0; nb=0
    for bi in batches(tr):
        e = tok([texts[i] for i in bi], truncation=True, max_length=192,
                padding=True, return_tensors="pt")
        ex = torch.tensor(XLs[bi], dtype=torch.float).cuda()
        y = torch.tensor(Y[bi], dtype=torch.float).cuda()
        opt2.zero_grad()
        loss = loss_fn2(model2(e.input_ids.cuda(), e.attention_mask.cuda(), extra=ex), y)
        loss.backward(); opt2.step(); tot+=loss.item(); nb+=1
    print(f"epoch {epoch+1}: loss {tot/nb:.4f}")

model2.eval(); probs=[]
with torch.no_grad():
    for bi in batches(te, shuffle=False):
        e = tok([texts[i] for i in bi], truncation=True, max_length=192,
                padding=True, return_tensors="pt")
        ex = torch.tensor(XLs[bi], dtype=torch.float).cuda()
        probs.append(torch.sigmoid(
            model2(e.input_ids.cuda(), e.attention_mask.cuda(), extra=ex)).cpu().numpy())
probC = np.vstack(probs)
mC = score(Y[te], probC)
print("=== C: combined ==="); [print(" ", k, v) for k,v in mC.items()]

# Ablation Table Summary

In [14]:
rows = []
for name, m in [("lexicon",mA),("DAPT",mB),("combined",mC)]:
    for dim, v in m.items():
        rows.append({"variant":name, "dimension":dim, **v})
abl = pd.DataFrame(rows)
print(abl.pivot_table(index="dimension", columns="variant", values="f1"))
abl.to_csv(f"{PROC}/ablation_results.csv", index=False)

variant      DAPT  combined  lexicon
dimension                           
bullying    0.531     0.452    0.257
financial   0.380     0.427    0.169
parasocial  0.311     0.127    0.265


# Saving the best model + predictions for Phase D

In [15]:
best = probC                       # set to whichever variant won
torch.save(model2.state_dict(), "models/harm_clf.pt")
pred = d.iloc[te][["uid","type","text","y_para","y_bully","y_fin"]].copy()
pred[["p_para","p_bully","p_fin"]] = best
pred.to_parquet(f"{PROC}/harm_predictions.parquet")
print("saved model + predictions")

saved model + predictions


# Auto documentaion of trial and error

In [5]:
import numpy as np, torch, pandas as pd, gc, time
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from clf import FocalLoss, Posts, HarmClassifier, score

tok = AutoTokenizer.from_pretrained("models/dapt_final")
texts = d.text.tolist()

def loader(idx, bs, shuffle=False):
    return DataLoader(Posts([texts[i] for i in idx], Y[idx], tok),
                      batch_size=bs, shuffle=shuffle)

def train_eval(lr, gamma, epochs, bs=16):
    torch.manual_seed(0); np.random.seed(0)
    model = HarmClassifier(AutoModel.from_pretrained("models/dapt_final")).cuda()
    pos = Y[tr].sum(0)
    pw = torch.tensor((len(tr)-pos)/np.maximum(pos,1), dtype=torch.float).cuda()
    loss_fn = FocalLoss(gamma=gamma, pos_weight=pw)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    dl = loader(tr, bs, shuffle=True)
    for e in range(epochs):
        model.train()
        for ids, mask, y in dl:
            opt.zero_grad()
            loss = loss_fn(model(ids.cuda(), mask.cuda()), y.cuda())
            loss.backward(); opt.step()
    model.eval(); probs=[]
    with torch.no_grad():
        for ids, mask, y in loader(te, bs):
            probs.append(torch.sigmoid(model(ids.cuda(), mask.cuda())).cpu().numpy())
    m = score(Y[te], np.vstack(probs))
    f1 = np.mean([m[k]["f1"] for k in m]); auc = np.mean([m[k]["auc"] for k in m])
    del model, opt; gc.collect(); torch.cuda.empty_cache()   # FREE GPU
    return f1, auc

trials = [
    (2e-5, 2.0, 4),
    (1e-5, 2.0, 4),
    (3e-5, 2.0, 4),
    (2e-5, 1.0, 4),
]
rows=[]
for lr, gamma, ep in trials:
    t0=time.time()
    f1, auc = train_eval(lr, gamma, ep)
    rows.append({"lr":lr,"gamma":gamma,"epochs":ep,"macro_f1":round(f1,3),"macro_auc":round(auc,3)})
    print(f"lr={lr} gamma={gamma} ep={ep} -> F1 {f1:.3f} AUC {auc:.3f}  ({time.time()-t0:.0f}s)")

sweep = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
sweep.to_csv("data/processed/hparam_sweep.csv", index=False)
print("\nBEST:", sweep.head(1).to_dict("records")[0])

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lr=2e-05 gamma=2.0 ep=4 -> F1 0.379 AUC 0.801  (84s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lr=3e-05 gamma=2.0 ep=4 -> F1 0.345 AUC 0.797  (82s)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lr=2e-05 gamma=1.0 ep=4 -> F1 0.398 AUC 0.805  (82s)

BEST: {'lr': 2e-05, 'gamma': 1.0, 'epochs': 4, 'macro_f1': 0.398, 'macro_auc': 0.805}


# Retraining best config on ALL data (for Phase D)

In [8]:
best = sweep.iloc[0]
LR, GAMMA, EP = float(best.lr), float(best.gamma), int(best.epochs)
print(f"final config: lr={LR} gamma={GAMMA} epochs={EP}")

all_idx = np.arange(len(d))
model = HarmClassifier(AutoModel.from_pretrained("models/dapt_final")).cuda()
pos = Y.sum(0); pw = torch.tensor((len(d)-pos)/np.maximum(pos,1), dtype=torch.float).cuda()
loss_fn = FocalLoss(gamma=GAMMA, pos_weight=pw)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
dl = loader(all_idx, 16, shuffle=True)
for e in range(EP):
    model.train(); tot=0
    for ids, mask, y in dl:
        opt.zero_grad()
        loss = loss_fn(model(ids.cuda(), mask.cuda()), y.cuda())
        loss.backward(); opt.step(); tot+=loss.item()
    print(f"epoch {e+1}: loss {tot/len(dl):.4f}")

torch.save(model.state_dict(), "models/harm_clf_final.pt")

# predict on all annotated posts for Phase D
model.eval(); probs=[]
with torch.no_grad():
    for ids, mask, y in loader(all_idx, 16):
        probs.append(torch.sigmoid(model(ids.cuda(), mask.cuda())).cpu().numpy())
P = np.vstack(probs)
out = d[["uid","type","text","y_para","y_bully","y_fin"]].copy()
out[["p_para","p_bully","p_fin"]] = P
out.to_parquet("data/processed/harm_predictions.parquet")
print("saved final model + full predictions")

final config: lr=2e-05 gamma=1.0 epochs=4


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: models/dapt_final
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: loss 0.5487
epoch 2: loss 0.4104
epoch 3: loss 0.3227
epoch 4: loss 0.2546
saved final model + full predictions


In [1]:
import os
os.chdir("/root/tf-project/venv/Projects/BehaviouralAnalysis")
print("final model:", os.path.exists("models/harm_clf_final.pt"))
print("predictions:", os.path.exists("data/processed/harm_predictions.parquet"))
import pandas as pd
print("prediction rows:", len(pd.read_parquet("data/processed/harm_predictions.parquet")))

final model: True
predictions: True
prediction rows: 2495


In [7]:
import torch
print("free VRAM:", (torch.cuda.get_device_properties(0).total_memory
      - torch.cuda.memory_allocated())/1e9, "GB")

free VRAM: 12.86012928 GB


best = probC                       # set to whichever variant won
torch.save(model2.state_dict(), "models/harm_clf.pt")
pred = d.iloc[te][["uid","type","text","y_para","y_bully","y_fin"]].copy()
pred[["p_para","p_bully","p_fin"]] = best
pred.to_parquet(f"{PROC}/harm_predictions.parquet")
print("saved model + predictions")